# LangGraph와 AgentCore 메모리 훅 (장기 메모리)

## 소개

이 노트북은 LangGraph 프레임워크를 사용하여 대화형 AI 에이전트에 Amazon Bedrock AgentCore 메모리 기능을 통합하는 방법을 보여줍니다. 여러 대화 세션에 걸친 **장기 메모리** 유지에 초점을 맞추어, 에이전트가 과거 상호작용에서 사용자 선호도, 식이 제한, 맥락 정보를 추출하고 기억할 수 있도록 합니다.

## 튜토리얼 세부사항

| 항목         | 세부내용                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                        |
| 에이전트 사용 사례       | 영양 어시스턴트                                                              |
| 에이전트 프레임워크   | LangGraph                                                                        |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                     |
| 튜토리얼 구성요소 | AgentCore 장기 메모리, 커스텀 메모리 전략, Pre/Post 모델 훅     |
| 예제 난이도  | 중급                                                                     |

학습 내용:
- UserPreference 커스텀 오버라이드 전략으로 AgentCore 메모리 생성
- 자동 메모리 저장 및 검색을 위한 pre/post 모델 훅 구현
- 세션 간 사용자 선호도를 기억하는 영양 어시스턴트 구축
- 시맨틱 검색을 사용한 관련 사용자 컨텍스트 검색
- 커스텀 메모리 추출 및 통합 프롬프트 설정

### 시나리오 컨텍스트

이 예제에서는 식이 제한, 좋아하는 음식, 요리 선호도, 건강 목표 등 여러 대화에 걸쳐 사용자 컨텍스트를 기억할 수 있는 **영양 어시스턴트**를 만듭니다. 에이전트는 대화에서 사용자 선호도를 자동으로 추출하고 저장한 후, 향후 상호작용에서 관련 컨텍스트를 검색하여 개인화된 영양 조언을 제공합니다.

## 아키텍처

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근 권한

환경 설정부터 시작하겠습니다!

In [ ]:
# https://github.com/langchain-ai/langchain-aws 에서 필요한 라이브러리 설치
%pip install -qr requirements.txt

In [ ]:
import os
import logging

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid


region = os.getenv("AWS_REGION", "us-east-1")
logging.getLogger("math-agent").setLevel(logging.DEBUG)

In [ ]:
# 스토어로 사용할 AgentCoreMemoryStore를 가져옵니다
from langgraph_checkpoint_aws import AgentCoreMemoryStore

# 이 예제에서는 컨텍스트 저장을 위해 InMemorySaver만 사용합니다.
# 프로덕션 환경에서는 메모리 스토어와 원활하게 작동하는 AgentCoreMemorySaver를 체크포인터로 사용하는 것을 강력히 권장합니다
# from langgraph_checkpoint_aws import AgentCoreMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from custom_memory_prompts import consolidation_prompt, extraction_prompt

In [ ]:
memory_name = "NutritionAssistant"
client = MemoryClient(region_name=region)
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant",
    memory_execution_role_arn="arn:aws:iam::YOUR_ACCOUNT:role/YOUR_ROLE",  # Please provide a role with a valid trust policy
    strategies=[
        {
            StrategyType.CUSTOM.value: {
                "name": "NutritionPreferences",
                "description": "Captures customer food preferences and behavior",
                "namespaces": ["/{actorId}/preferences/"],
                "configuration": {
                    "userPreferenceOverride": {
                        "extraction": {
                            "appendToPrompt": extraction_prompt,
                            "modelId": MODEL_ID,
                        },
                        "consolidation": {
                            "appendToPrompt": consolidation_prompt,
                            "modelId": MODEL_ID,
                        },
                    }
                },
            }
        },
    ],
)
memory_id = memory["id"]

### 메모리 설정 개요

AgentCore 메모리 설정에는 다음이 포함됩니다:

- **커스텀 전략**: 대화에서 영양 선호도를 추출합니다
- **네임스페이스**: 사용자별로 메모리를 구성합니다 (`{actorId}/preferences/`)
- **커스텀 프롬프트**: 음식 선호도를 위한 특수 추출 및 통합 로직
- **모델 통합**: 메모리 처리에 Claude 3.7 Sonnet을 사용합니다

메모리 시스템은 대화를 자동으로 처리하여 일시적이거나 관련 없는 정보를 필터링하면서 지속적인 사용자 선호도를 추출합니다.

## 3단계: 메모리 스토어 및 LLM 초기화

이제 AgentCore 메모리 스토어와 언어 모델을 초기화합니다.

In [ ]:
# 장기 메모리 저장 및 검색을 활성화하기 위해 스토어를 초기화합니다
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Bedrock LLM 초기화
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

## 4단계: 메모리 훅 구현

메모리 저장 및 검색을 자동으로 처리하기 위해 pre 및 post 모델 훅을 생성합니다:

- **Pre 모델 훅**: LLM 호출 전 관련 사용자 선호도(시맨틱 검색 기반)를 검색하고 컨텍스트를 추가합니다
- **Post 모델 훅**: 장기 메모리 추출을 위해 대화 메시지를 저장합니다

### 메모리 처리 작동 방식

1. 메시지가 actor_id 및 session_id와 함께 AgentCore 메모리에 저장됩니다
2. 커스텀 전략이 대화를 처리하여 영양 선호도를 추출합니다
3. 추출된 선호도가 `{actorId}/preferences/` 네임스페이스에 저장됩니다
4. 향후 대화에서 관련 선호도를 검색하고 조회할 수 있습니다

**참고**: LangChain 메시지 타입은 장기 메모리로 적절히 추출될 수 있도록 스토어 내부에서 AgentCore 메모리 메시지 타입으로 자동 변환됩니다.

In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """LLM 호출 전에 실행되어 최신 사용자 메시지를 저장하는 훅"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # 런타임에 얻는 actor와 session 조합에 메시지를 저장합니다
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # LLM 호출 전 마지막 사용자 메시지를 저장합니다
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break
    # 마지막 메시지를 기반으로 사용자 선호도를 검색하고 상태에 추가합니다
    user_preferences_namespace = (actor_id, "preferences/")
    preferences = store.search(user_preferences_namespace, query=msg.content, limit=5)

    # 현재 메시지 앞에 컨텍스트를 추가하기 위한 AI 메시지를 생성합니다
    if preferences:
        context_items = [pref.value for pref in preferences]
        context_message = AIMessage(
            content=f"[User Context: {', '.join(str(item) for item in context_items)}]"
        )
        # 마지막 사용자 메시지 앞에 컨텍스트 메시지를 삽입합니다
        return {"messages": messages[:-1] + [context_message, messages[-1]]}

    return {"llm_input_messages": messages}


def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """LLM 호출 후에 실행되어 최신 사용자 메시지를 저장하는 훅"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # 런타임에 얻는 actor와 session 조합에 메시지를 저장합니다
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # LLM의 응답을 AgentCore 메모리에 저장합니다
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    return {"messages": messages}

## 5단계: LangGraph 에이전트 생성

이제 메모리 훅이 통합된 LangGraph의 `create_react_agent`를 사용하여 영양 어시스턴트 에이전트를 만듭니다. 도구 노드에는 장기 메모리 검색 도구만 포함되며, pre 및 post 모델 훅이 인수로 지정됩니다.

**참고**: 커스텀 에이전트 구현에서는 이 패턴에 따라 모든 워크플로우에 필요한 대로 Store와 도구를 설정할 수 있습니다. Pre/post 모델 훅을 사용하거나, 전체 대화를 마지막에 저장하는 등의 방식이 가능합니다.

In [ ]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[],  # 이 예제에서는 추가 도구가 필요 없습니다
    checkpointer=InMemorySaver(),  # 대화 상태 관리용
    pre_model_hook=pre_model_hook,  # LLM 호출 전 사용자 선호도를 검색합니다
    post_model_hook=post_model_hook,  # LLM 응답 후 대화를 저장합니다
)

## 6단계: 에이전트 런타임 설정

사용자와 세션에 대한 고유 식별자로 에이전트를 설정해야 합니다. 이 ID들은 메모리 구성 및 검색에 중요합니다.

### 그래프 호출 입력
최신 사용자 메시지만 `inputs` 인수로 전달하면 됩니다. 다른 상태 변수도 포함할 수 있지만, 간단한 `create_react_agent`에서는 메시지만 필요합니다.

### LangGraph RuntimeConfig
LangGraph에서 config는 호출 시 필요한 속성(예: 사용자 ID 또는 세션 ID)을 포함하는 `RuntimeConfig`입니다. `AgentCoreMemorySaver`에서는 config에 `thread_id`와 `actor_id`를 설정해야 합니다. 예를 들어, AgentCore 호출 엔드포인트에서 호출자의 ID 또는 사용자 ID를 기반으로 이를 할당할 수 있습니다. 추가 [문서는 여기](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)에서 확인할 수 있습니다.

In [ ]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1",  # 필수: 내부적으로 Bedrock AgentCore session_id에 매핑됩니다
        "actor_id": actor_id,  # 필수: 내부적으로 Bedrock AgentCore actor_id에 매핑됩니다
    }
}

## 7단계: 에이전트 테스트

음식 선호도에 대한 대화를 통해 영양 어시스턴트를 테스트해 봅시다. 에이전트는 향후 사용을 위해 사용자 선호도를 자동으로 추출하고 저장합니다.

In [ ]:
# 실행 중 에이전트 출력을 예쁘게 출력하는 헬퍼 함수
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # 이미 출력한 메시지인지 확인합니다
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


# 오늘 밤 좋아하는 식사 중 하나인 연어와 밥, 채소를 요리합니다. 역도 대회 준비를 위한 좋은 매크로 영양소 구성입니다.
# 이 요리의 맛을 더 좋게 하고 단백질과 비타민 섭취를 개선할 수 있는 방법을 물어봅니다.
prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)

### 무엇이 저장되었나요?
보시다시피, 모델은 아직 우리의 선호도나 식이 제한에 대한 인사이트가 없습니다.

이 pre/post 모델 훅 구현에서는 두 개의 메시지가 저장되었습니다. 사용자의 첫 번째 메시지와 AI 모델의 응답이 모두 AgentCore 메모리에 대화 이벤트로 저장되었습니다. 장기 메모리가 추출되기까지 몇 초가 걸릴 수 있으므로, 처음에 아무것도 찾지 못하면 몇 초 후에 다시 시도해 보세요.

이 메시지들은 사실 및 사용자 선호도 네임스페이스에서 AgentCore 장기 메모리로 추출되었습니다. 실제로 스토어를 직접 확인하여 지금까지 저장된 내용을 검증할 수 있습니다:

In [ ]:
# 사용자 선호도 네임스페이스를 검색합니다
search_namespace = (actor_id, "preferences/")
result = store.search(search_namespace, query="food", limit=3)
print(f"선호도 네임스페이스 결과: {result}")

### 에이전트의 스토어 접근

**참고** - AgentCore 메모리는 백그라운드에서 이러한 이벤트를 처리하므로, 메모리가 추출되고 장기 메모리 검색에 임베딩되기까지 몇 초가 걸릴 수 있습니다.

좋습니다! 이전 대화의 메시지를 기반으로 네임스페이스에 장기 메모리가 추출되었음을 확인했습니다.

이제 새 세션을 시작하고 저녁 식사로 무엇을 요리할지 추천을 요청해 봅시다. 에이전트는 스토어를 사용하여 추출된 장기 메모리에 접근하여 사용자가 확실히 좋아할 추천을 제공할 수 있습니다.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # 새 세션 ID
        "actor_id": actor_id,  # 동일한 actor ID
    }
}

# 새로운 날, 오늘 저녁 무엇을 만들어야 할지 물어봅니다
run_agent("Today's a new day, what should I make for dinner tonight?", config)

### 마무리

보시다시피, 에이전트는 사용자 선호도 네임스페이스 검색에서 pre 모델 훅 컨텍스트를 받았고, 사실 네임스페이스에서 장기 메모리를 자체적으로 검색하여 사용자에게 포괄적인 답변을 만들 수 있었습니다.

AgentCoreMemoryStore는 매우 유연하며, pre/post 모델 훅 또는 스토어 작업이 포함된 도구 자체를 포함하여 다양한 방식으로 구현할 수 있습니다. 체크포인팅을 위한 AgentCoreMemorySaver와 함께 사용하면, 전체 대화 상태와 장기 인사이트를 결합하여 복잡하고 지능적인 에이전트 시스템을 구성할 수 있습니다.